# Projected GMM Metric Validation

This notebook checks the metric functions from `prototyping/metrics_synthetic_controlled.py` without training a model. The goal is to make the measurement behavior visible.

We build true projected-GMM data in two regimes:

- `D = d = 2`: the ambient space is the true latent space.
- `D = 32, d = 2`: samples live near a planted 2D subspace inside a high-dimensional ambient space.

Then we compare true reference samples against controlled test samples with increasing noise. Quality metrics should worsen as noise grows. Memorization metrics are different: they are not generic quality scores, so we validate them with a separate copy-to-training probe.

In [ ]:
from pathlib import Path
import math
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mem_par_metric_validation_mpl")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import torch

cwd = Path.cwd().resolve()
repo_root = None
for candidate in (cwd, *cwd.parents):
    if (candidate / "src").is_dir() and (candidate / "prototyping").is_dir():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the mem_par checkout.")

src_root = repo_root / "src"
for path in (repo_root, src_root):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from mp_config import get_base_config
from prototyping.metrics_synthetic_controlled import (
    memorization_metrics_in_spaces,
    oracle_gmm_metrics,
    project_to_subspace,
)

torch.set_default_dtype(torch.float32)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

out_dir = repo_root / "tmp" / "metric_validation_figures"
out_dir.mkdir(parents=True, exist_ok=True)

print("repo_root:", repo_root)
print("figure output:", out_dir)

## Metric meanings used in this notebook

- `oracle_latent_nll_mean`: lower is better. It asks whether samples land in high-density regions of the known latent GMM.
- `oracle_full_nll_mean`: lower is better. It adds the off-subspace Gaussian penalty, so it should react sharply to high-D residual noise.
- `off_subspace_energy_fraction` and `residual_norm_mean`: lower is better. They ask whether samples stay near the planted subspace.
- `latent_frechet_distance_to_oracle`: lower is better. It is the FID-like mean/covariance check in true latent coordinates.
- `latent_sliced_w2`: lower is better. It compares generated and reference latent clouds through many 1D projections.
- `component_weight_l1`: lower is better. It checks whether component responsibilities are balanced.
- `f_mem` and `mean_nn_ratio`: these are memorization probes, not quality probes. Exact training copies should give high `f_mem` and low ratio; true fresh samples should not.

In [ ]:
def make_metric_config(ratio_threshold=0.3):
    config = get_base_config()
    config.evaluation.ratio_threshold = float(ratio_threshold)
    return config


def make_sliced_directions(d, n_directions=128, seed=0):
    generator = torch.Generator().manual_seed(int(seed))
    directions = torch.randn(n_directions, d, generator=generator)
    return directions / directions.norm(dim=1, keepdim=True).clamp(min=1.0e-12)


def make_projected_gmm(
    *,
    D,
    d,
    K=3,
    n_train=96,
    n_reference=768,
    sigma_z=0.10,
    sigma_eta=1.0e-3,
    seed=0,
    geometry_seed=0,
):
    geometry_generator = torch.Generator().manual_seed(int(geometry_seed))
    basis, _ = torch.linalg.qr(torch.randn(D, d, generator=geometry_generator))
    basis = basis[:, :d].contiguous()

    if d == 2:
        angles = torch.linspace(0.0, 2.0 * math.pi, K + 1)[:-1]
        means = 2.0 * torch.stack([torch.cos(angles), torch.sin(angles)], dim=1)
    else:
        means = 2.0 * torch.randn(K, d, generator=geometry_generator)

    def sample(n, sample_seed):
        generator = torch.Generator().manual_seed(int(sample_seed))
        labels = torch.randint(K, (n,), generator=generator)
        latent = means[labels] + sigma_z * torch.randn(n, d, generator=generator)
        ambient = latent @ basis.T + sigma_eta * torch.randn(n, D, generator=generator)
        return ambient.contiguous(), latent.contiguous(), labels

    train, train_latent, train_labels = sample(n_train, seed)
    reference, reference_latent, reference_labels = sample(n_reference, seed + 1)
    directions = make_sliced_directions(d, seed=seed + 1009)

    return {
        "D": D,
        "d": d,
        "K": K,
        "sigma_z": sigma_z,
        "sigma_eta": sigma_eta,
        "basis": basis,
        "means": means.contiguous(),
        "train": train,
        "train_latent": train_latent,
        "train_labels": train_labels,
        "reference": reference,
        "reference_latent": reference_latent,
        "reference_labels": reference_labels,
        "directions": directions,
        "config": make_metric_config(),
    }


def metric_summary(samples, data):
    n = int(samples.shape[0])
    reference_latent = data["reference_latent"][:n]
    oracle = oracle_gmm_metrics(
        samples,
        data["basis"],
        data["means"],
        data["sigma_z"],
        data["sigma_eta"],
        reference_latent,
        data["directions"],
    )
    mem = memorization_metrics_in_spaces(
        samples,
        data["train"],
        data["basis"],
        data["train_latent"],
        chunk_size=512,
        config=data["config"],
    )
    return {**oracle, **mem}


def corrupt_reference(data, *, n_eval, noise_scale, kind, seed):
    base = data["reference"][:n_eval]
    basis = data["basis"]
    sigma_z = float(data["sigma_z"])
    generator = torch.Generator().manual_seed(int(seed))
    parts = project_to_subspace(base, basis)

    if kind == "latent":
        latent_noise = noise_scale * sigma_z * torch.randn(parts["latent"].shape, generator=generator)
        latent = parts["latent"] + latent_noise
        return latent @ basis.T + parts["residual"]

    if kind == "off_subspace":
        raw = torch.randn(base.shape, generator=generator)
        raw_parts = project_to_subspace(raw, basis)
        residual_noise = raw_parts["residual"]
        residual_noise = residual_noise / residual_noise.pow(2).mean().sqrt().clamp(min=1.0e-12)
        return base + noise_scale * sigma_z * residual_noise

    if kind == "ambient":
        return base + noise_scale * sigma_z * torch.randn(base.shape, generator=generator)

    raise ValueError(f"Unknown corruption kind: {kind}")


def train_copies(data, *, n_eval, noise_scale, seed):
    generator = torch.Generator().manual_seed(int(seed))
    indices = torch.randint(data["train"].shape[0], (n_eval,), generator=generator)
    copied = data["train"][indices]
    noise = noise_scale * float(data["sigma_z"]) * torch.randn(copied.shape, generator=generator)
    return copied + noise


QUALITY_METRICS = [
    "oracle_latent_nll_mean",
    "oracle_full_nll_mean",
    "latent_frechet_distance_to_oracle",
    "latent_sliced_w2",
    "off_subspace_energy_fraction",
    "residual_norm_mean",
    "component_weight_l1",
]

MEMORY_METRICS = [
    "ambient_f_mem",
    "ambient_mean_nn_ratio",
    "latent_f_mem",
    "latent_mean_nn_ratio",
]

In [ ]:
def run_noise_sweep(*, D, d, kind, noise_levels, n_repeats=8, n_eval=512):
    metrics = QUALITY_METRICS + MEMORY_METRICS
    true_values = {metric: [] for metric in metrics}
    test_values = {metric: np.zeros((n_repeats, len(noise_levels)), dtype=np.float64) for metric in metrics}

    for rep in range(n_repeats):
        data = make_projected_gmm(D=D, d=d, seed=1000 + 17 * rep, geometry_seed=2000 + 19 * rep)
        true_samples = data["reference"][:n_eval]
        true_metrics = metric_summary(true_samples, data)
        for metric in metrics:
            true_values[metric].append(float(true_metrics[metric]))

        for col, noise_scale in enumerate(noise_levels):
            test_samples = corrupt_reference(
                data,
                n_eval=n_eval,
                noise_scale=float(noise_scale),
                kind=kind,
                seed=3000 + 101 * rep + col,
            )
            test_metrics = metric_summary(test_samples, data)
            for metric in metrics:
                test_values[metric][rep, col] = float(test_metrics[metric])

    return {
        "D": D,
        "d": d,
        "kind": kind,
        "noise_levels": np.asarray(noise_levels, dtype=np.float64),
        "true": {metric: np.asarray(values, dtype=np.float64) for metric, values in true_values.items()},
        "test": test_values,
    }


def run_copy_sweep(*, D, d, noise_levels, n_repeats=8, n_eval=512):
    metrics = MEMORY_METRICS + ["oracle_latent_nll_mean", "oracle_full_nll_mean"]
    true_values = {metric: [] for metric in metrics}
    test_values = {metric: np.zeros((n_repeats, len(noise_levels)), dtype=np.float64) for metric in metrics}

    for rep in range(n_repeats):
        data = make_projected_gmm(D=D, d=d, seed=4000 + 17 * rep, geometry_seed=5000 + 19 * rep)
        true_samples = data["reference"][:n_eval]
        true_metrics = metric_summary(true_samples, data)
        for metric in metrics:
            true_values[metric].append(float(true_metrics[metric]))

        for col, noise_scale in enumerate(noise_levels):
            test_samples = train_copies(
                data,
                n_eval=n_eval,
                noise_scale=float(noise_scale),
                seed=6000 + 101 * rep + col,
            )
            test_metrics = metric_summary(test_samples, data)
            for metric in metrics:
                test_values[metric][rep, col] = float(test_metrics[metric])

    return {
        "D": D,
        "d": d,
        "kind": "train_copy",
        "noise_levels": np.asarray(noise_levels, dtype=np.float64),
        "true": {metric: np.asarray(values, dtype=np.float64) for metric, values in true_values.items()},
        "test": test_values,
    }


def plot_metric_grid(result, metrics, title, filename, ncols=3):
    noise = result["noise_levels"]
    nrows = int(math.ceil(len(metrics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.2 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    for ax, metric in zip(axes_flat, metrics):
        test = result["test"][metric]
        true = result["true"][metric]
        mean = test.mean(axis=0)
        std = test.std(axis=0)
        true_mean = true.mean()
        true_std = true.std()

        ax.plot(noise, mean, marker="o", label="test mean")
        ax.fill_between(noise, mean - std, mean + std, alpha=0.18, label="test +/- std")
        ax.axhline(true_mean, color="black", linestyle="--", linewidth=1.2, label="true reference mean")
        ax.axhspan(true_mean - true_std, true_mean + true_std, color="black", alpha=0.08)
        ax.set_title(metric)
        ax.set_xlabel("extra noise scale x sigma_z")
        ax.set_ylabel("metric value")

        if metric in {"oracle_full_nll_mean", "off_subspace_energy_fraction", "residual_norm_mean"}:
            ax.set_yscale("symlog", linthresh=1.0e-4)

    for ax in axes_flat[len(metrics):]:
        ax.axis("off")

    handles, labels = axes_flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
    fig.suptitle(title, y=1.02, fontsize=14)
    fig.tight_layout()
    path = out_dir / filename
    fig.savefig(path, bbox_inches="tight")
    print("saved", path)
    return fig


noise_levels = np.array([0.0, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0], dtype=np.float64)

## Visual checks: true reference versus noisy test samples

The first plot is the easy 2D case. The second plot is high-D: we look at the true latent projection and the residual norm, because ambient scatter is not meaningful in 32 dimensions.

In [ ]:
def plot_visual_examples():
    data_2d = make_projected_gmm(D=2, d=2, seed=11, geometry_seed=22)
    true_2d = data_2d["reference"][:512]
    means_2d = data_2d["means"] @ data_2d["basis"].T

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.7))
    for ax, scale in zip(axes, [0.0, 2.0, 8.0]):
        test = corrupt_reference(data_2d, n_eval=512, noise_scale=scale, kind="latent", seed=123 + int(scale * 10))
        ax.scatter(true_2d[:, 0], true_2d[:, 1], s=8, alpha=0.22, label="true reference")
        ax.scatter(test[:, 0], test[:, 1], s=8, alpha=0.50, label="test")
        ax.scatter(means_2d[:, 0], means_2d[:, 1], s=80, marker="x", linewidths=2.0, label="means")
        ax.set_title(f"2D latent noise scale {scale:g}")
        ax.set_aspect("equal", adjustable="box")
    axes[0].legend(loc="best")
    fig.tight_layout()
    path = out_dir / "visual_2d_true_vs_noisy_test.png"
    fig.savefig(path, bbox_inches="tight")
    print("saved", path)

    data_hd = make_projected_gmm(D=32, d=2, seed=33, geometry_seed=44)
    true_hd = data_hd["reference"][:512]
    off_test = corrupt_reference(data_hd, n_eval=512, noise_scale=4.0, kind="off_subspace", seed=777)
    true_latent = project_to_subspace(true_hd, data_hd["basis"])["latent"]
    test_latent = project_to_subspace(off_test, data_hd["basis"])["latent"]
    true_residual = project_to_subspace(true_hd, data_hd["basis"])["residual_norm"]
    test_residual = project_to_subspace(off_test, data_hd["basis"])["residual_norm"]

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    axes[0].scatter(true_latent[:, 0], true_latent[:, 1], s=8, alpha=0.24, label="true reference")
    axes[0].scatter(test_latent[:, 0], test_latent[:, 1], s=8, alpha=0.50, label="off-subspace test")
    axes[0].scatter(data_hd["means"][:, 0], data_hd["means"][:, 1], s=80, marker="x", linewidths=2.0, label="means")
    axes[0].set_title("High-D latent projection")
    axes[0].set_aspect("equal", adjustable="box")
    axes[0].legend(loc="best")

    axes[1].hist(true_residual.numpy(), bins=40, alpha=0.55, label="true reference")
    axes[1].hist(test_residual.numpy(), bins=40, alpha=0.55, label="off-subspace test")
    axes[1].set_title("High-D residual norm")
    axes[1].set_xlabel("||x - (x @ B) B.T||")
    axes[1].legend(loc="best")
    fig.tight_layout()
    path = out_dir / "visual_highd_latent_and_residual.png"
    fig.savefig(path, bbox_inches="tight")
    print("saved", path)


plot_visual_examples()

## Quality metric sweeps

The dashed line is the mean metric value for true reference samples. The solid line is the mean for controlled test samples as noise increases. The shaded bands are one standard deviation across repeated random seeds.

In [ ]:
result_2d_latent = run_noise_sweep(D=2, d=2, kind="latent", noise_levels=noise_levels)
fig = plot_metric_grid(
    result_2d_latent,
    QUALITY_METRICS,
    "2D GMM: latent noise makes test samples less like the true GMM",
    "quality_2d_latent_noise.png",
)
plt.show()

In [ ]:
result_highd_latent = run_noise_sweep(D=32, d=2, kind="latent", noise_levels=noise_levels)
fig = plot_metric_grid(
    result_highd_latent,
    QUALITY_METRICS,
    "High-D projected GMM: latent noise hurts latent quality metrics",
    "quality_highd_latent_noise.png",
)
plt.show()

In [ ]:
result_highd_off = run_noise_sweep(D=32, d=2, kind="off_subspace", noise_levels=noise_levels)
fig = plot_metric_grid(
    result_highd_off,
    QUALITY_METRICS,
    "High-D projected GMM: off-subspace noise mostly hits residual and full NLL",
    "quality_highd_off_subspace_noise.png",
)
plt.show()

## Memorization probe

Memorization metrics should not be interpreted as generic sample quality. Here the test samples are direct training copies plus increasing noise. At zero copy noise, the nearest-neighbor ratio should be tiny and `f_mem` should be high. As copy noise grows, those memorization signals should relax toward the true-reference baseline.

In [ ]:
copy_noise_levels = np.array([0.0, 0.05, 0.10, 0.25, 0.5, 1.0, 2.0, 4.0], dtype=np.float64)

copy_2d = run_copy_sweep(D=2, d=2, noise_levels=copy_noise_levels)
fig = plot_metric_grid(
    copy_2d,
    MEMORY_METRICS + ["oracle_latent_nll_mean"],
    "2D memorization probe: exact training copies are detected",
    "memorization_2d_train_copies.png",
    ncols=3,
)
plt.show()

copy_highd = run_copy_sweep(D=32, d=2, noise_levels=copy_noise_levels)
fig = plot_metric_grid(
    copy_highd,
    MEMORY_METRICS + ["oracle_latent_nll_mean", "oracle_full_nll_mean"],
    "High-D memorization probe: exact training copies are detected",
    "memorization_highd_train_copies.png",
    ncols=3,
)
plt.show()

## Read the plots this way

1. In 2D latent-noise plots, `oracle_latent_nll_mean`, `latent_frechet_distance_to_oracle`, and `latent_sliced_w2` should rise as test samples move away from the planted GMM.
2. In high-D latent-noise plots, the same latent metrics should worsen, while residual metrics should not be the main signal.
3. In high-D off-subspace plots, `oracle_full_nll_mean`, `off_subspace_energy_fraction`, and `residual_norm_mean` should worsen strongly, while latent-only metrics can stay close to the true baseline. That is the point of separating latent likelihood from full ambient likelihood.
4. In copy-to-training plots, memorization metrics should flag exact copies. If noisy copies become less exact, `f_mem` falls and nearest-neighbor ratios rise. That is useful, but it is a different question from sample quality.